# 01. Tiền xử lý & Khám phá Dữ liệu (Data Cleaning & EDA)

**Thành viên phụ trách:** Thành viên 1 - Data Engineering & Preprocessing

**Mục tiêu quy trình xử lý dữ liệu thực tế (`1.csv`):**
1. **Team Win%:** Tách bảng tỷ lệ thắng của từng đội bóng theo mùa giải (`team_win_pct`).
2. **Lọc kỷ nguyên hiện đại (1980–2026):** Giữ các mùa giải NBA từ năm 1980 trở đi (khi vạch 3 điểm bắt đầu áp dụng) và loại trừ mùa giải 1999 (bị hủy All-Star do Lockout).
3. **Tạo nhãn All-Star & Xử lý Trade giữa mùa:** Kết hợp nhãn `replaced` và `IS_ALLSTAR`; xác định đội bóng chính (`main_tm`) thi đấu nhiều trận nhất và ưu tiên giữ dòng tổng hợp (`TOT`, `2TM`, `3TM`, `4TM`, `5TM`) để mỗi cặp `(player_id, season)` là duy nhất.
4. **Ghép tỷ lệ thắng:** Ghép `team_win_pct` tương ứng với đội bóng chính của cầu thủ.
5. **Bảo toàn lịch sử danh tiếng (`prev_allstar_count`):** Thống kê số lần được chọn All-Star tích lũy trước mùa giải hiện tại (`cumsum().shift(1)`) trước bước lọc nhiễu.
6. **Lọc nhiễu & Xử lý Missing Value:** Lọc các cầu thủ có $G \ge 15$ trận và $MP/G \ge 10.0$ phút/trận (phát hiện 4 ngoại lệ chấn thương lịch sử: Kobe Bryant 2014, Grant Hill 2001, Yao Ming 2011, Alonzo Mourning 2001). Điền giá trị 0.0 cho các tỷ lệ ném thiếu.
7. **Era Adjustment (Chuẩn hóa Z-Score theo mùa):** Tính Z-score nội bộ từng mùa giải cho 6 chỉ số: `pts_per_game`, `ast_per_game`, `trb_per_game`, `per`, `ws`, `vorp`.
8. **Phân chia dữ liệu:** Lưu trữ `data/processed/nba_clean.csv` ($15.542 \times 34$) và chia thành `train.csv` ($\le 2016$), `val.csv` ($2017 - 2019$), `test.csv` ($\ge 2020$).
9. **Trực quan hóa EDA:** Vẽ 4 biểu đồ phân tích và lưu vào `figures/eda/`.

In [ ]:
import sys
import os
from pathlib import Path

# Thêm thư mục gốc dự án vào sys.path để import từ src
ROOT_DIR = Path.cwd().resolve().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình font hiển thị tiếng Việt trên đồ thị
plt.rcParams['font.sans-serif'] = ['Segoe UI', 'Arial', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="whitegrid", font="Segoe UI")

from src.preprocessing import clean_raw_nba_data, split_by_season, build_preprocessing_pipeline

print("Imports thành công từ src.preprocessing!")

## Bước 1: Nạp file thô `data/raw/1.csv`
Tải file dữ liệu gốc ban đầu ($55.018$ dòng $\times$ $409$ cột).

In [ ]:
raw_csv_path = ROOT_DIR / "data" / "raw" / "1.csv"
print(f"Đọc dữ liệu từ: {raw_csv_path}")

df_raw = pd.read_csv(raw_csv_path, low_memory=False)
print(f"Kích thước dữ liệu thô ban đầu: {df_raw.shape}")
df_raw[['player_id', 'player', 'season', 'team', 'g', 'pts_per_game']].head(5)

## Bước 2: Tách bảng tỷ lệ thắng của đội bóng (Team Win%)
Trích xuất tỷ lệ thắng `w / (w + l)` của từng đội bóng trong từng mùa giải từ năm 1980 trở đi.

In [ ]:
team_rows = df_raw[
    df_raw['player_id'].isnull() & 
    df_raw['abbreviation'].notnull() & 
    df_raw['w'].notnull() & 
    (df_raw['lg'] == 'NBA') & 
    (df_raw['season'] >= 1980)
].copy()

team_rows['team_win_pct'] = team_rows['w'] / (team_rows['w'] + team_rows['l'])
team_win_lookup = team_rows[['season', 'abbreviation', 'team_win_pct']].drop_duplicates(subset=['season', 'abbreviation'])
print(f"Số lượng bản ghi tỷ lệ thắng đội bóng: {len(team_win_lookup)}")
team_win_lookup.head()

## Bước 3: Lọc cầu thủ NBA từ mùa 1980 (loại trừ mùa 1999 Lockout)
Loại bỏ mùa giải 1998–1999 do cuộc đình công khiến trận All-Star bị hủy, tránh việc mô hình học nhầm các siêu sao mùa đó bị trượt.

In [ ]:
players = df_raw[
    df_raw['player_id'].notnull() & 
    df_raw['pts_per_game'].notnull() & 
    (df_raw['lg'] == 'NBA') & 
    (df_raw['season'] >= 1980) &
    (df_raw['season'] != 1999)
].copy()

print(f"Số bản ghi cầu thủ sau lọc mùa giải: {len(players)}")

## Bước 4: Tạo nhãn All_Star & Xử lý cầu thủ đổi đội giữa mùa (Deduplicate Traded Players)
- Gộp `replaced` và `IS_ALLSTAR` thành nhãn `All_Star`.
- Tìm đội bóng thi đấu nhiều trận nhất làm `main_tm`.
- Ưu tiên giữ dòng tổng (`TOT`, `2TM`, `3TM`, `4TM`, `5TM`) để mỗi cặp `(player_id, season)` là duy nhất.

In [ ]:
players['All_Star'] = np.where((players['replaced'].notnull()) | (players['IS_ALLSTAR'] == 1.0), 1, 0)
players['All_Star'] = players.groupby(['player_id', 'season'])['All_Star'].transform('max')

multi_tm_codes = ['TOT', '2TM', '3TM', '4TM', '5TM']
real_team_rows = players[~players['team'].isin(multi_tm_codes)].copy()
main_team = (
    real_team_rows.sort_values('g', ascending=False)
    .drop_duplicates(subset=['player_id', 'season'], keep='first')[['player_id', 'season', 'team']]
    .rename(columns={'team': 'main_tm'})
)

players['is_total_row'] = np.where(players['team'].isin(multi_tm_codes), 0, 1)
players = (
    players.sort_values(['player_id', 'season', 'is_total_row', 'g'], ascending=[True, True, True, False])
    .drop_duplicates(subset=['player_id', 'season'], keep='first')
    .copy()
)

print(f"Số dòng cầu thủ sau khi khử trùng lặp trade: {len(players)}")

## Bước 5: Ghép tỷ lệ thắng đội & Bảo toàn lịch sử danh tiếng (`prev_allstar_count`)
- Ghép `team_win_pct` qua `main_tm`.
- Tính số lần All-Star tích lũy trước mùa giải hiện tại bằng `cumsum().shift(1)` trước bước lọc nhiễu, đảm bảo không mất lịch sử của siêu sao từng bị chấn thương.

In [ ]:
# Ghép team_win_pct
players = pd.merge(players, main_team, on=['player_id', 'season'], how='left')
players['main_tm'] = players['main_tm'].fillna(players['team'])
players = pd.merge(players, team_win_lookup, left_on=['season', 'main_tm'], right_on=['season', 'abbreviation'], how='left')

# Tính số lần All-Star quá khứ
players['season'] = players['season'].astype(int)
players = players.sort_values(['player_id', 'season']).reset_index(drop=True)
players['prev_allstar_count'] = (
    players.groupby('player_id')['All_Star']
    .transform(lambda x: x.cumsum().shift(1).fillna(0))
    .astype(int)
)
print("Phân bố số lần All-Star quá khứ (Top 5 giá trị phổ biến):")
print(players['prev_allstar_count'].value_counts().head())

## Bước 6: Lọc nhiễu ($G \ge 15, MP/G \ge 10.0$) & Phát hiện 4 ngoại lệ lịch sử
Áp dụng ngưỡng tối thiểu để loại bỏ các cầu thủ thi đấu quá ít làm nhiễu tỷ lệ ném rổ:
- Trước lọc: 1.182 lượt All-Star.
- Sau lọc: 1.178 lượt All-Star (loại đúng 4 trường hợp chấn thương nặng nghỉ gần hết mùa: Kobe Bryant 2014, Grant Hill 2001, Yao Ming 2011, Alonzo Mourning 2001).

In [ ]:
as_before = players[players['All_Star'] == 1]
as_removed = as_before[(as_before['g'] < 15) | (as_before['mp_per_game'] < 10.0)]
print(f"Số lượng All-Star trước khi lọc nhiễu: {len(as_before)}")
print(f"4 ngoại lệ chấn thương lịch sử bị lọc:")
print(as_removed[['player', 'season', 'g', 'mp_per_game', 'replaced']])

players_clean = players[(players['g'] >= 15) & (players['mp_per_game'] >= 10.0)].copy()

for pct_col in ['fg_percent', 'x3p_percent', 'ft_percent', 'ts_percent']:
    players_clean[pct_col] = players_clean[pct_col].fillna(0.0)

players_clean['gs'] = players_clean['gs'].fillna(0).astype(int)
players_clean['g'] = players_clean['g'].astype(int)
players_clean['age'] = players_clean['age'].astype(int)

print(f"\nSố lượng cầu thủ sau khi lọc nhiễu: {len(players_clean)}")

## Bước 7: Chuẩn hóa Z-Score theo từng mùa giải (Era Adjustment)
Tính Z-Score trong nội bộ từng mùa giải cho 6 chỉ số: `pts_per_game`, `ast_per_game`, `trb_per_game`, `per`, `ws`, `vorp` để khử chênh lệch thời kỳ (pace, scoring shifts qua các thập niên).

In [ ]:
zscore_features = ['pts_per_game', 'ast_per_game', 'trb_per_game', 'per', 'ws', 'vorp']
for col in zscore_features:
    mean = players_clean.groupby('season')[col].transform('mean')
    std = players_clean.groupby('season')[col].transform('std').replace(0, 1)
    players_clean[f'{col}_zscore'] = ((players_clean[col] - mean) / std).fillna(0.0)

final_cols = [
    'player_id', 'player', 'season', 'pos', 'age', 'team', 'main_tm',
    'g', 'gs', 'mp_per_game', 'pts_per_game', 'ast_per_game', 'trb_per_game',
    'stl_per_game', 'blk_per_game', 'tov_per_game',
    'fg_percent', 'x3p_percent', 'ft_percent',
    'per', 'ts_percent', 'usg_percent', 'ws', 'bpm', 'vorp',
    'team_win_pct', 'prev_allstar_count',
    'pts_per_game_zscore', 'ast_per_game_zscore', 'trb_per_game_zscore',
    'per_zscore', 'ws_zscore', 'vorp_zscore',
    'All_Star'
]

nba_clean = players_clean[final_cols].copy().reset_index(drop=True)
print(f"Kích thước tập dữ liệu sạch: {nba_clean.shape}")
print(f"Số lượng giá trị khuyết thiếu (NaN): {nba_clean.isna().sum().sum()}")
print(f"Phân bố nhãn All_Star: \n{nba_clean['All_Star'].value_counts()}")

## Bước 8: Lưu trữ `data/processed/nba_clean.csv` và Phân chia Train/Validation/Test
- **Train:** Mùa giải $\le 2016$ ($11.473$ mẫu, $912$ All-Stars, $7,95\%$)
- **Validation:** $2017 - 2019$ ($1.187$ mẫu, $80$ All-Stars, $6,74\%$)
- **Test:** Mùa giải $\ge 2020$ ($2.882$ mẫu, $186$ All-Stars, $6,45\%$)

In [ ]:
processed_dir = ROOT_DIR / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

# 1. Lưu toàn bộ dữ liệu sạch
clean_csv_path = processed_dir / "nba_clean.csv"
nba_clean.to_csv(clean_csv_path, index=False)
print(f"Đã lưu thành công {clean_csv_path}")

# 2. Phân chia theo thời gian
train_df, val_df, test_df = split_by_season(nba_clean, train_end=2016, val_end=2019, season_col="season")

train_df.to_csv(processed_dir / "train.csv", index=False)
val_df.to_csv(processed_dir / "val.csv", index=False)
test_df.to_csv(processed_dir / "test.csv", index=False)

print(f"Train set: {train_df.shape} (Mùa {train_df['season'].min()} - {train_df['season'].max()}), All-Stars: {train_df['All_Star'].sum()} ({train_df['All_Star'].mean():.2%})")
print(f"Val set:   {val_df.shape} (Mùa {val_df['season'].min()} - {val_df['season'].max()}), All-Stars: {val_df['All_Star'].sum()} ({val_df['All_Star'].mean():.2%})")
print(f"Test set:  {test_df.shape} (Mùa {test_df['season'].min()} - {test_df['season'].max()}), All-Stars: {test_df['All_Star'].sum()} ({test_df['All_Star'].mean():.2%})")

## Bước 9: Trực quan hóa Phân tích Khám phá Dữ liệu (EDA)
Tạo 4 biểu đồ phân tích và lưu vào `figures/eda/`:
- **Biểu đồ 1:** Phân bố mất cân bằng lớp (Class Imbalance).
- **Biểu đồ 2:** Phân tách điểm số trung bình (pts_per_game KDE).
- **Biểu đồ 3:** Ma trận tương quan giữa các chỉ số nâng cao và All_Star.
- **Biểu đồ 4:** Boxplot phân phối Season Z-Scores.

In [ ]:
eda_dir = ROOT_DIR / "figures" / "eda"
eda_dir.mkdir(parents=True, exist_ok=True)

# 1. Biểu đồ mất cân bằng lớp
plt.figure(figsize=(7, 4.5))
counts = nba_clean['All_Star'].value_counts()
ax = sns.barplot(x=['Non All-Star (0)', 'All-Star (1)'], y=counts.values, hue=['Non All-Star (0)', 'All-Star (1)'], palette=['#4A90E2', '#E94E77'], legend=False)
plt.title("Phân bố nhãn All-Star (Mất cân bằng lớp: 7.58% vs 92.42%)", fontsize=12, fontweight='bold')
plt.ylabel("Số lượng mẫu", fontsize=10)
for p in ax.patches:
    h = p.get_height()
    pct = h / len(nba_clean) * 100
    ax.annotate(f"{int(h):,} ({pct:.2f}%)", (p.get_x() + p.get_width() / 2., h / 2),
                ha='center', va='center', color='white', fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig(eda_dir / "01_class_imbalance.png", dpi=300)
plt.show()

# 2. Phân tách điểm số
plt.figure(figsize=(8, 4.5))
sns.kdeplot(data=nba_clean[nba_clean['All_Star'] == 0]['pts_per_game'], label='Non All-Star (Median ~ 8.8)', fill=True, color='#4A90E2', alpha=0.5)
sns.kdeplot(data=nba_clean[nba_clean['All_Star'] == 1]['pts_per_game'], label='All-Star (Median ~ 21.8)', fill=True, color='#E94E77', alpha=0.6)
plt.title("Sự phân tách về điểm số trung bình (pts_per_game)", fontsize=12, fontweight='bold')
plt.xlabel("Điểm số trung bình mỗi trận (PTS/G)", fontsize=10)
plt.ylabel("Mật độ phân phối (Density)", fontsize=10)
plt.legend(frameon=True)
plt.tight_layout()
plt.savefig(eda_dir / "02_pts_distribution.png", dpi=300)
plt.show()